# Smart MCQ Solver — BiLSTM + Attention (from scratch, PyTorch)

Word-order-aware sequence model, built up in the same 12 sections used to explain it:
data preprocessing, tokenization, vocabulary, padding, Dataset/DataLoader, embedding, BiLSTM,
attention, classification head, training loop, validation, MAP@3 inference.

Unlike TF-IDF, this model reads token order -- word position, negation scope, which phrase modifies which
-- none of which a bag-of-words representation can capture at all.


### Setup: device safety + config


In [15]:
!pip install -q --no-deps wandb

import re, random
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def get_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        (torch.zeros(1, device="cuda") + 1).cpu()
        return torch.device("cuda")
    except Exception as e:
        print("CUDA unusable, falling back to CPU:", e)
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

CONFIG = {
    "max_vocab_size": 30000, "min_freq": 2, "max_len": 60,
    "embed_dim": 128, "hidden_dim": 128, "num_layers": 1,
    "dropout": 0.3, "label_smoothing": 0.05,
    "lr": 1e-3, "weight_decay": 1e-4, "batch_size": 64,
    "epochs": 40, "early_stop_min_epochs": 12, "early_stop_patience": 12,
    "early_stop_min_delta": 0.0001, "val_split": 0.15, "grad_clip": 1.0,
}

wandb.login()
wandb.init(project="smart-mcq-solver", name="bilstm-attention", config=CONFIG)


CUDA unusable, falling back to CPU: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Using device: cpu


## 1. Data preprocessing
**Theory:** TF-IDF is order-blind. A BiLSTM reads tokens in sequence, so word order now carries meaning --
preprocessing choices matter more here than they did for TF-IDF.

**Why it matters:** an unseen word becomes the same `<UNK>` token for every occurrence, unlike TF-IDF where
it's just a harmless zero in a 40000-dim vector -- so preprocessing consistency between train and test
directly controls how much vocabulary is usable versus wasted on `<UNK>`.

**Common mistake:** stripping words like "not" as if they were stopwords can flip an MCQ's meaning; a
sequence model that's supposed to use word order is especially sensitive to this.


In [16]:
train_df = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv")
LABELS = ["A", "B", "C", "D", "E"]
label2idx = {l: i for i, l in enumerate(LABELS)}

def build_text(df):
    text = (
        "question: " + df["prompt"].fillna("") +
        " choices: [a] " + df["A"].fillna("") + " [b] " + df["B"].fillna("") +
        " [c] " + df["C"].fillna("") + " [d] " + df["D"].fillna("") + " [e] " + df["E"].fillna("")
    )
    return text.str.lower()

train_df["text"]  = build_text(train_df)
test_df["text"]   = build_text(test_df)
train_df["label"] = train_df["answer"].map(label2idx)
train_df[["text", "label"]].head(3)


,text,label
0,question: pick the best possible answer: what ...,1
1,question: what is accelerator-based light-ion ...,0
2,question: determine the correct option: what i...,2


## 2. Tokenization
**Theory:** splits a string into the discrete units the model reasons over. Simple word-level regex
tokenization is used instead of a heavier library -- fully transparent, easy to explain line by line.

**Why necessary:** `nn.Embedding` needs integer ids, not strings.

**Common mistake:** `text.split()` leaves punctuation glued to words (`"mars?"` != `"mars"`), silently
doubling the vocabulary with near-duplicates. A regex over alphanumeric runs avoids this.


In [17]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text)   # strips punctuation, keeps alphanumeric tokens

print(tokenize("Which planet is called the Red Planet?".lower()))


['which', 'planet', 'is', 'called', 'the', 'red', 'planet']


## 3. Vocabulary building
**Theory:** fixed mapping `token -> integer id`. `<PAD>=0` and `<UNK>=1` are always reserved.

**Why necessary:** the embedding table is a `(vocab_size, embed_dim)` matrix -- every token needs a fixed
row index into it, decided before training starts.

**Important distinction from the TF-IDF notebooks:** there we fit the vectorizer on train+test text, which
is fine since TF-IDF only needs to know which words exist. Here we build vocabulary from **train only** --
more honest (mirrors genuinely new test data), and means `<UNK>` actually gets exercised as intended.

**Common mistake:** skipping `min_freq` filtering lets typos and one-off tokens bloat the vocabulary with
entries the model only ever sees once and can never learn a good embedding for.


In [18]:
def build_vocab(texts, max_vocab_size, min_freq):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token, freq in counter.most_common():
        if freq < min_freq or len(vocab) >= max_vocab_size:
            continue
        vocab[token] = len(vocab)
    return vocab

vocab = build_vocab(train_df["text"], CONFIG["max_vocab_size"], CONFIG["min_freq"])
PAD_IDX, UNK_IDX = vocab["<PAD>"], vocab["<UNK>"]
print("Vocab size:", len(vocab))


Vocab size: 2973


## 4. Sequence padding
**Theory:** fills every sequence to a fixed `max_len` with `<PAD>` (or truncates), producing a rectangular
tensor. The **true length** (before padding) is tracked separately.

**Why necessary:** batches must be rectangular tensors -- variable-length lists can't be stacked.

**Common mistake:** forgetting to track true lengths means the LSTM can't tell real tokens from padding --
this especially breaks the backward direction of a BiLSTM, which would start by reading padding as if it
were the sequence's real starting point.


In [19]:
def encode(text, vocab, max_len):
    tokens = tokenize(text)[:max_len]
    length = len(tokens)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]
    ids += [PAD_IDX] * (max_len - length)          # pad on the right
    return ids, max(length, 1)                      # guard against a fully-empty sequence

ids, length = encode("which planet is red", vocab, CONFIG["max_len"])
print(ids[:8], "...", "true length:", length)


[18, 409, 5, 773, 0, 0, 0, 0] ... true length: 4


## 5. Dataset and DataLoader
**Theory:** standard PyTorch pattern -- `Dataset.__getitem__` returns one example, `DataLoader` batches,
shuffles, and parallelizes loading.

**Shapes:** `__getitem__` returns `(ids: (max_len,), length: scalar, label: scalar)`; after batching:
`(batch, max_len)`, `(batch,)`, `(batch,)`.

**Common mistakes:** token ids must be `LongTensor` (embedding lookup requires int64, not float). Shuffling
the test loader breaks the correspondence between predictions and question IDs at submission time.


In [20]:
class MCQDataset(Dataset):
    def __init__(self, texts, vocab, max_len, labels=None):
        self.texts, self.vocab, self.max_len, self.labels = texts, vocab, max_len, labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length = encode(self.texts.iloc[idx], self.vocab, self.max_len)
        ids = torch.tensor(ids, dtype=torch.long)
        length = torch.tensor(length, dtype=torch.long)
        if self.labels is not None:
            return ids, length, torch.tensor(self.labels.iloc[idx], dtype=torch.long)
        return ids, length

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"], train_df["label"], test_size=CONFIG["val_split"], random_state=SEED, stratify=train_df["label"]
)

train_loader = DataLoader(MCQDataset(train_texts, vocab, CONFIG["max_len"], train_labels), batch_size=CONFIG["batch_size"], shuffle=True)
val_loader   = DataLoader(MCQDataset(val_texts, vocab, CONFIG["max_len"], val_labels),     batch_size=CONFIG["batch_size"], shuffle=False)
test_loader  = DataLoader(MCQDataset(test_df["text"], vocab, CONFIG["max_len"]),           batch_size=CONFIG["batch_size"], shuffle=False)


## 6. Embedding layer
**Theory:** `nn.Embedding(vocab_size, embed_dim)` is a learnable lookup table of shape `(vocab_size,
embed_dim)`. Unlike TF-IDF's fixed hand-computed weights, these vectors are learned via backprop.

**Why necessary:** a raw integer id has no numerical meaning the network should use -- embedding turns a
meaningless index into a dense vector it can do arithmetic on.

**Shapes:** `(batch, seq_len)` int64 -> `(batch, seq_len, embed_dim)` float32.

**Common mistake (easy to miss):** forgetting `padding_idx=PAD_IDX`. Without it, `<PAD>` gets its own
trainable embedding and wastes model capacity learning a representation for "nothing".


In [21]:
class WordEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

    def forward(self, x):
        return self.embedding(x)   # (batch, seq_len) -> (batch, seq_len, embed_dim)


## 7. BiLSTM
**Theory:** an LSTM keeps a cell state `c_t` (slow-changing memory) and hidden state `h_t` (per-step
output), controlled by three gates:

```
f_t = sigma(W_f . [h_(t-1), x_t] + b_f)     # forget gate
i_t = sigma(W_i . [h_(t-1), x_t] + b_i)     # input gate
o_t = sigma(W_o . [h_(t-1), x_t] + b_o)     # output gate
c~_t = tanh(W_c . [h_(t-1), x_t] + b_c)      # candidate values
c_t = f_t * c_(t-1) + i_t * c~_t             # ADDITIVE cell update -- this is why gradients don't vanish
h_t = o_t * tanh(c_t)
```

**Bidirectional** = run the LSTM left-to-right and right-to-left, concatenate at each position:
`h_t = [h_forward_t ; h_backward_t]`. A word late in an option can inform how an earlier question word is
represented, and vice versa -- plain left-to-right can't do that.

**Why necessary:** this is where the model reasons about sequence -- word order, negation scope -- none of
which TF-IDF can represent.

**Shapes:** `(batch, seq_len, embed_dim)` -> `(batch, seq_len, 2*hidden_dim)`.

**Common mistakes:** forgetting the `2x` when sizing the next layer (classic shape-mismatch bug); not using
`pack_padded_sequence`, which makes the LSTM waste compute on `<PAD>` tokens and, worse, makes the backward
pass start from padding instead of the sequence's real end.


In [22]:
class BiLSTMEncoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,   # PyTorch warns if dropout>0 with 1 layer
        )

    def forward(self, x, lengths):
        # pack -> LSTM skips <PAD> positions and correctly finds each sequence's real start for the backward pass
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        return out   # (batch, seq_len, 2*hidden_dim)


## 8. Attention layer
**Theory:** using only the BiLSTM's final hidden state forces the whole sequence through one fixed-size
vector -- an information bottleneck, worse for longer questions. Additive attention instead scores every
token and takes a learned weighted sum:

```
e_t = v^T . tanh(W . h_t + b)     # scalar relevance score per token
alpha_t = softmax(e_t) over t      # normalize into weights summing to 1
context = sum_t(alpha_t * h_t)     # weighted sum -> one vector
```

**Why necessary:** gives the model learned, content-dependent pooling instead of a fixed rule (e.g. mean
pooling) -- it can decide per-example which tokens matter most.

**Shapes:** `(batch, seq_len, 2*hidden_dim)` -> scores `(batch, seq_len)` -> weights `(batch, seq_len)` ->
context `(batch, 2*hidden_dim)`.

**Common mistake (the big one for attention):** forgetting to mask padding positions before the softmax --
`<PAD>` tokens can otherwise receive nonzero attention weight via the scorer's bias term, especially early
in training. Fix: set padding positions' scores to a large negative number before softmax.


In [23]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out, lengths):
        # lstm_out: (batch, seq_len, hidden_dim) where hidden_dim here = 2*lstm_hidden_dim
        scores = self.v(torch.tanh(self.attn(lstm_out))).squeeze(-1)   # (batch, seq_len)

        # mask: True where position is real, False where it's padding
        seq_len = lstm_out.size(1)
        mask = torch.arange(seq_len, device=lengths.device)[None, :] < lengths[:, None]
        scores = scores.masked_fill(~mask, -1e9)   # padding positions -> ~0 weight after softmax

        weights = torch.softmax(scores, dim=1)                          # (batch, seq_len)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)  # (batch, hidden_dim)
        return context, weights


## 9. Classification head
**Theory:** small feedforward block mapping the pooled context vector to 5 class logits -- same pattern as
the TF-IDF notebooks' final layers.

**Why necessary:** the BiLSTM+attention stack produces a rich vector representation; something still has to
turn that into "how likely is each of A/B/C/D/E."

**Shapes:** `(batch, 2*hidden_dim)` -> `(batch, 5)` raw logits.

**Common mistake:** applying softmax manually before `CrossEntropyLoss`, which already applies
`log_softmax` internally in a numerically stable way.


In [24]:
class ClassifierHead(nn.Module):
    def __init__(self, hidden_dim, n_classes=5, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, n_classes),   # raw logits
        )

    def forward(self, context):
        return self.net(context)

class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, dropout, n_classes=5):
        super().__init__()
        self.embedding = WordEmbedding(vocab_size, embed_dim, pad_idx)
        self.encoder = BiLSTMEncoder(embed_dim, hidden_dim, num_layers, dropout)
        self.attention = Attention(hidden_dim * 2)      # *2 for bidirectional
        self.classifier = ClassifierHead(hidden_dim * 2, n_classes, dropout)

    def forward(self, x, lengths):
        embedded = self.embedding(x)                                # (batch, seq_len, embed_dim)
        lstm_out = self.encoder(embedded, lengths)                   # (batch, seq_len, 2*hidden_dim)
        context, attn_weights = self.attention(lstm_out, lengths)    # (batch, 2*hidden_dim)
        logits = self.classifier(context)                            # (batch, 5)
        return logits, attn_weights

model = BiLSTMAttentionClassifier(
    vocab_size=len(vocab), embed_dim=CONFIG["embed_dim"], hidden_dim=CONFIG["hidden_dim"],
    num_layers=CONFIG["num_layers"], pad_idx=PAD_IDX, dropout=CONFIG["dropout"],
).to(device)
wandb.watch(model, log="all", log_freq=50)
print(model)


BiLSTMAttentionClassifier(
  (embedding): WordEmbedding(
    (embedding): Embedding(2973, 128, padding_idx=0)
  )
  (encoder): BiLSTMEncoder(
    (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  )
  (attention): Attention(
    (attn): Linear(in_features=256, out_features=256, bias=True)
    (v): Linear(in_features=256, out_features=1, bias=False)
  )
  (classifier): ClassifierHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.3, inplace=False)
      (3): Linear(in_features=128, out_features=5, bias=True)
    )
  )
)


## 10. Training loop
**Theory/why:** same forward -> loss -> backward -> step cycle as the TF-IDF notebooks, plus one
RNN-specific addition: **gradient clipping**. Backpropagating through many timesteps can make gradients
explode (the mirror image of the vanishing-gradient problem the LSTM gates solve) -- clipping caps the
gradient norm so one bad batch can't blow up the weights.

**Common mistakes:** skipping gradient clipping (meaningfully more important here than in the feedforward
TF-IDF models); computing `lengths` after padding instead of before.


In [25]:
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

def map3_score(logits, targets):
    top3 = torch.topk(logits, k=3, dim=1).indices
    score = 0.0
    for pred_row, true in zip(top3.tolist(), targets.tolist()):
        if true in pred_row:
            score += 1.0 / (pred_row.index(true) + 1)
    return score / len(targets)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = total_acc = total_map3 = 0.0
    n_batches = 0

    for ids, lengths, y in loader:
        ids, lengths, y = ids.to(device), lengths.to(device), y.to(device)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                logits, _ = model(ids, lengths)
                loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])  # cap exploding gradients
                scaler.step(optimizer)
                scaler.update()

        preds = logits.argmax(dim=1)
        total_loss += loss.item()
        total_acc  += (preds == y).float().mean().item()
        total_map3 += map3_score(logits.detach(), y)
        n_batches  += 1

    return total_loss / n_batches, total_acc / n_batches, total_map3 / n_batches


## 11. Validation
**Theory/why:** `model.eval()` disables dropout (every neuron participates, matching real inference) --
important here since dropout sits in the classifier head and, if `num_layers>1`, between LSTM layers.
`torch.no_grad()` skips gradient bookkeeping since we're not updating weights.

**Common mistake:** forgetting `model.eval()` before validation -- dropout would keep randomly zeroing
neurons during evaluation too, making val MAP@3 noisier and more pessimistic than the model actually is.


In [26]:
best_val_map3 = 0.0
patience_left = CONFIG["early_stop_patience"]

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_acc, train_map3 = run_epoch(train_loader, train=True)
    val_loss,   val_acc,   val_map3   = run_epoch(val_loader,   train=False)
    scheduler.step(val_map3)

    wandb.log({
        "epoch": epoch, "train/loss": train_loss, "train/map3": train_map3,
        "val/loss": val_loss, "val/map3": val_map3, "lr": optimizer.param_groups[0]["lr"],
    })

    if val_map3 > best_val_map3 + CONFIG["early_stop_min_delta"]:
        best_val_map3 = val_map3
        patience_left = CONFIG["early_stop_patience"]
        torch.save(model.state_dict(), "best_bilstm.pt")
    elif epoch >= CONFIG["early_stop_min_epochs"]:
        patience_left -= 1

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} val_loss {val_loss:.4f} | val_map3 {val_map3:.4f} | patience {patience_left}")
    if epoch >= CONFIG["early_stop_min_epochs"] and patience_left <= 0:
        print("Early stopping: no val_map3 improvement for", CONFIG["early_stop_patience"], "epochs (after warmup).")
        break

print("Best val MAP@3:", best_val_map3)


Epoch 01 | train_loss 1.5786 val_loss 1.4940 | val_map3 0.5351 | patience 12
Epoch 02 | train_loss 1.0250 val_loss 0.6074 | val_map3 0.9255 | patience 12
Epoch 03 | train_loss 0.3768 val_loss 0.2724 | val_map3 1.0000 | patience 12
Epoch 04 | train_loss 0.2586 val_loss 0.2428 | val_map3 1.0000 | patience 12
Epoch 05 | train_loss 0.2421 val_loss 0.2337 | val_map3 1.0000 | patience 12
Epoch 06 | train_loss 0.2354 val_loss 0.2298 | val_map3 1.0000 | patience 12
Epoch 07 | train_loss 0.2341 val_loss 0.2288 | val_map3 1.0000 | patience 12
Epoch 08 | train_loss 0.2324 val_loss 0.2283 | val_map3 1.0000 | patience 12
Epoch 09 | train_loss 0.2323 val_loss 0.2278 | val_map3 1.0000 | patience 12
Epoch 10 | train_loss 0.2316 val_loss 0.2273 | val_map3 1.0000 | patience 12
Epoch 11 | train_loss 0.2314 val_loss 0.2273 | val_map3 1.0000 | patience 12
Epoch 12 | train_loss 0.2311 val_loss 0.2268 | val_map3 1.0000 | patience 11
Epoch 13 | train_loss 0.2309 val_loss 0.2266 | val_map3 1.0000 | patience 10

## 12. MAP@3 inference
**Theory/why:** identical logic to the TF-IDF notebooks -- `topk(logits, k=3)` on raw logits, since softmax
is monotonic and doesn't change the ranking, so there's no need to compute it explicitly at inference time.


In [27]:
model.load_state_dict(torch.load("best_bilstm.pt"))
model.eval()

all_top3 = []
with torch.no_grad():
    for ids, lengths in test_loader:
        ids, lengths = ids.to(device), lengths.to(device)
        logits, _ = model(ids, lengths)
        top3 = torch.topk(logits, k=3, dim=1).indices.cpu().tolist()
        all_top3.extend(top3)

pred_labels = [" ".join(LABELS[i] for i in row) for row in all_top3]
submission = pd.DataFrame({"ID": test_df["id"], "Prediction": pred_labels})
submission.to_csv("submission_bilstm.csv", index=False)
submission.head()


,ID,Prediction
0,1,A E C
1,2,B A D
2,3,B E D
3,4,E C B
4,5,C A D


In [28]:
wandb.finish()


epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
lr,██████▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁
train/loss,█▅▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/map3,▁▅█████████████████████
val/loss,█▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/map3,▁▇█████████████████████
epoch,23
lr,3e-05
train/loss,0.22992
train/map3,1
val/loss,0.22603
